# Feature Extraction

Combining existing features to build brand new ones that capture the same information more efficiently.

The new features never existed in the raw data — the originals are discarded.

| | Selection | Extraction |
|---|---|---|
| Output | Keeps original columns | Creates brand new columns |
| Interpretability | High | Low — components are abstract |

---
# Part 1 — Linear Methods

Find new axes by combining original features in **straight-line** ways.
* Relationship between features follows direct or indirect proportion so if one goes up, the other goes up (or down) at a constant rate. No curves, no sudden jumps. One columne affecting the other column.

| Method | Needs Labels | Best For |
|---|---|---|
| PCA | No | Correlated features, general compression |
| LDA | Yes | Maximising class separation |
| ICA | No | Separating mixed independent signals |

#### Example Data

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

data = {
    'homework':   [85, 60, 92, 45, 78, 55],
    'quiz_1':     [88, 55, 95, 50, 72, 60],
    'quiz_2':     [84, 62, 91, 48, 76, 57],
    'midterm':    [90, 58, 93, 52, 74, 63],
    'attendance': [95, 70, 88, 60, 82, 65],
}
df = pd.DataFrame(data, index=['Alice', 'Bob', 'Carol', 'Dan', 'Eve', 'Frank'])
y = [1, 0, 1, 0, 1, 0]  # 1 = pass,  0 = fail  (used by LDA only)

df

,homework,quiz_1,quiz_2,midterm,attendance
Alice,85,88,84,90,95
Bob,60,55,62,58,70
Carol,92,95,91,93,88
Dan,45,50,48,52,60
Eve,78,72,76,74,82
Frank,55,60,57,63,65


In [5]:
# All linear methods are scale sensitive so standardise first
X = StandardScaler().fit_transform(df)

### 1. PCA — Principal Component Analysis

Rotates the data so the **axis of greatest spread** comes first (PC1), then the next most (PC2), and so on. Keeps the top *x* and discards the rest.

`(x - mean) / std  →  rotate axes  →  keep top x`
1. StandardScaler (standardise first)
2. rotate axes: PCA finds the new directions
3. keep top x:  keep only the components you want

Use when you have many correlated features and want to compress them.

In [6]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

pd.DataFrame(X_pca, columns=['component_1', 'component_2'], index=df.index).round(2)

,component_1,component_2
Alice,2.50,0.19
Bob,-1.50,0.34
Carol,2.92,-0.38
Dan,-2.97,-0.14
Eve,0.73,0.30
Frank,-1.68,-0.32


In [7]:
# How much of the original information do the 2 components keep?
print('Per component:', pca.explained_variance_ratio_.round(3))
print('Total kept:   ', round(pca.explained_variance_ratio_.sum(), 3))
# 2 components captured 97% of the information from 5 original features

Per component: [0.972 0.017]
Total kept:    0.989


In [8]:
# Loadings — contribution of each original feature to each component
pd.DataFrame(
    pca.components_.T,
    index=df.columns,
    columns=['component_1', 'component_2']
).round(2)

# component_1: all exassessment columns = 0.45 :captures overall assessment performance
# component_2: attendance dmoninates = 0.69    :captures attendance vs score gap

,component_1,component_2
homework,0.45,0.18
quiz_1,0.45,-0.56
quiz_2,0.45,0.11
midterm,0.45,-0.41
attendance,0.44,0.69


### 2. LDA — Linear Discriminant Analysis

Finds the direction that **best separates the classes**. Unlike PCA (which ignores labels), LDA uses `y` to pull classes apart as far as possible.

What LDA does:
* maximise gap between classes — push the pass group and fail group as far apart as possible
* minimise spread within each class — keep each group's own values tight together

Use when classification is the goal and you have labels.

In [ ]:
lda = LinearDiscriminantAnalysis(n_components=1)
X_lda = lda.fit_transform(X, y)

pd.DataFrame(X_lda, columns=['discriminant_1'], index=df.index).round(2)
# Positive: likely pass    
# Negative: likely fail

,discriminant_1
Alice,2.84
Bob,-1.41
Carol,2.62
Dan,-3.67
Eve,1.45
Frank,-1.82


  PCA finds max variance — no labels needed. LDA finds max class separation — labels required.

### 3. ICA — Independent Component Analysis

Separates a mixed signal into independent underlying sources. 

Example: two people talking at the same time on one microphone — ICA pulls each voice apart.

`find weights W so that outputs are as statistically independent as possible`

Use when features are mixtures of independent signals (e.g. audio, EEG brain signals).

In [10]:
ica = FastICA(n_components=2, random_state=42, max_iter=1000)
X_ica = ica.fit_transform(X)

pd.DataFrame(X_ica, columns=['signal_1', 'signal_2'], index=df.index).round(2)
# Each signal represents one independent underlying source

,signal_1,signal_2
Alice,1.05,0.77
Bob,-0.81,1.10
Carol,1.46,-1.16
Dan,-1.29,-0.62
Eve,0.21,1.07
Frank,-0.64,-1.16


 PCA finds max variance — components can still correlate. ICA finds max independence — components cannot correlate.

---
---
# Part 2 — Non-Linear Methods

Linear methods only work when data can be separated by a **straight line**.

**Imagine**: two interlocking half-moons — no straight line can split them. Non-linear methods bend the space to handle this.

| Method | Best For | Use as Model Input |
|---|---|---|
| Kernel PCA | Non-linear structure in tabular data | Yes |
| t-SNE | Visualising high-dimensional clusters | No |
| Isomap | Data that lies on a curved surface | Yes |

#### Example Data

In [11]:
from sklearn.datasets import make_moons
from sklearn.decomposition import KernelPCA
from sklearn.manifold import TSNE, Isomap

# Two curved groups: a straight line cannot separate them
X_moons, y_moons = make_moons(n_samples=8, noise=0.05, random_state=42)

df_shape = pd.DataFrame(X_moons, columns=['x', 'y'])
df_shape['group'] = y_moons
df_shape.round(2)

,x,y,group
0,0.49,0.85,0
1,0.58,-0.33,1
2,0.98,0.03,0
3,1.98,0.48,1
4,-0.49,0.77,0
5,-0.09,0.47,1
6,-1.05,0.02,0
7,1.45,-0.44,1


In [18]:
# Add a noisy 3rd dimension(column) so extraction is meaningful
np.random.seed(42)
X3 = np.column_stack([X_moons, np.random.normal(0, 0.1, 8)])
X3_scaled = StandardScaler().fit_transform(X3)


### 1. Kernel PCA

Regular PCA can only find **straight-line** directions. Kernel PCA first maps the data into a higher-dimensional curved space, then runs PCA there — so it can capture patterns a straight line would miss.

Common kernels: `rbf` (default) · `poly` · `cosine`

In [13]:
kpca = KernelPCA(n_components=2, kernel='rbf')
X_kpca = kpca.fit_transform(X3_scaled)

df_kpca = pd.DataFrame(X_kpca, columns=['component_1', 'component_2'])
df_kpca['group'] = y_moons
df_kpca.round(2)
# Points in the same group should have similar component values

,component_1,component_2,group
0,0.43,0.08,0
1,-0.33,-0.44,1
2,-0.43,-0.22,0
3,-0.26,0.50,1
4,0.70,-0.13,0
5,0.59,-0.25,1
6,-0.10,0.72,0
7,-0.60,-0.26,1


### 2. t-SNE

Keeps nearby points **close together** in 2D. Excellent at revealing hidden clusters in high-dimensional data.

**Visualisation only** — numbers change every run, no stable meaning. Never use as model input.

`perplexity` controls how many neighbours each point considers — try values between 3 and 50.

In [14]:
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
X_tsne = tsne.fit_transform(X3_scaled)

df_tsne = pd.DataFrame(X_tsne, columns=['dim_1', 'dim_2'])
df_tsne['group'] = y_moons
df_tsne.round(2)
# The two groups should appear clearly separated

,dim_1,dim_2,group
0,37.080002,-79.489998,0
1,39.130001,33.500000,1
2,17.730000,4.670000,0
3,48.689999,-19.650000,1
4,26.590000,-109.970001,0
5,7.400000,-94.000000,1
6,-16.230000,-17.650000,0
7,8.360000,28.629999,1


---
### 3. Isomap

Measures distance **along the surface of the data**, not in a straight line through empty space.

Think of it as measuring by road rather than as the crow flies — the route follows the shape of the data.

Use when data lies on a curved surface (e.g. facial expressions, hand gestures).

`n_neighbors` sets how many nearby points to connect when tracing the surface.

In [15]:
iso = Isomap(n_components=2, n_neighbors=3)
X_iso = iso.fit_transform(X3_scaled)

df_iso = pd.DataFrame(X_iso, columns=['dim_1', 'dim_2'])
df_iso['group'] = y_moons
df_iso.round(2)
# Group 0 and group 1 should land on opposite sides

,dim_1,dim_2,group
0,-0.67,-0.00,0
1,0.48,-1.00,1
2,0.96,0.58,0
3,1.85,-0.30,1
4,-1.51,-1.16,0
5,-1.71,-0.76,1
6,-1.14,2.48,0
7,1.74,0.16,1


t-SNE preserves local clusters. Isomap preserves the global shape of the data.

---
# Summary

| Method | Type | Needs Labels | Model Input |
|---|---|---|---|
| PCA | Linear | No | Yes |
| LDA | Linear | Yes | Yes |
| ICA | Linear | No | Yes |
| Kernel PCA | Non-linear | No | Yes |
| t-SNE | Non-linear | No | No |
| Isomap | Non-linear | No | Yes |

# The End